# Auric CV assignment: Colab training (B0, B1)

Runtime: **T4 GPU (16 GB), Python 3.13** (Runtime > Change runtime type > T4 GPU). Colab's preinstalled CUDA torch is used as-is; only the other requirements are installed.
* Data is copied from Drive to `/content` and unzipped there (never trained from Drive). The image cache (`cache: disk`) is written next to the images on `/content`, not on Drive.
* Every run folder (config, command, env incl. GPU type, checkpoints, metrics) lives on Drive under `DRIVE_RUNS`. `last.pt` is written there **every epoch**, so a disconnect loses at most one epoch, and `epoch010.pt … epoch050.pt` are kept. **To resume: re-run the setup cells, then the same training cell.**
* Both runs: 50 epochs, SGD lr0 0.01 momentum 0.937 (explicit, not `auto`), AMP (fp16), seed 0, deterministic.
* Tiles for B1 are rebuilt deterministically in `/content/work` after a restart (same seed, same files).
* **Order:** setup → tests → (timing probe, done) → B1 train → B1 eval + checkpoint curve → B0 train → B0 eval + checkpoint curve.

In [ ]:
# ---- edit these ----
DRIVE_DATA = "/content/drive/MyDrive/auric/cv_dataset.zip"  # a .zip OR a folder containing train/ val/ classmap.txt
DRIVE_RUNS = "/content/drive/MyDrive/auric/runs"            # run folders persist here
REPO_URL   = "https://github.com/arukh281/auric_cv_assignment.git"
BRANCH     = "main"
TOKEN_SECRET = "GH_TOKEN"  # Colab secret (key icon) holding a GitHub token; only needed if the repo is private
DATA_ROOT, WORK_DIR, REPO_DIR = "/content/data", "/content/work", "/content/repo"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_RUNS, exist_ok=True)
assert os.path.exists(DRIVE_DATA), f"dataset not found on Drive: {DRIVE_DATA}"
print("dataset on Drive:", DRIVE_DATA, round(os.path.getsize(DRIVE_DATA) / 2**30, 2) if os.path.isfile(DRIVE_DATA) else "(folder)", "GiB")
!nvidia-smi

In [ ]:
# Clone (or update) the repo
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get(TOKEN_SECRET)
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # public repo or no secret set
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# Install non-torch requirements WITHOUT touching Colab's preinstalled CUDA torch (Python 3.13 runtime).
# The constraints file pins torch/torchvision/numpy/opencv to what Colab already has, so pip cannot replace them.
import sys, subprocess
assert sys.version_info[:2] >= (3, 13), f"expected Colab Python 3.13+, got {sys.version}"
frz = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()
keep = [l for l in frz if l.split("==")[0].lower() in ("torch", "torchvision", "numpy", "opencv-python", "opencv-python-headless")]
open("/content/colab_constraints.txt", "w").write("\n".join(keep) + "\n")
print("constraints:", keep)
!pip install -q -r requirements.txt -c /content/colab_constraints.txt
import torch, ultralytics
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU", torch.cuda.get_device_name(0), "| ultralytics", ultralytics.__version__)
# full lock of the environment actually used (each run folder also gets its own pip freeze + hardware.json)
!pip freeze > {DRIVE_RUNS}/colab_env_lock.txt

In [ ]:
# Copy dataset from Drive to local disk and unzip; generate data.yaml if missing
import os, shutil, subprocess, sys
from pathlib import Path
if not (Path(DATA_ROOT) / "_READY").exists():
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    if DRIVE_DATA.endswith(".zip"):
        subprocess.run(["cp", DRIVE_DATA, "/content/dataset.zip"], check=True)
        subprocess.run(["unzip", "-q", "/content/dataset.zip", "-d", DATA_ROOT], check=True)
        os.remove("/content/dataset.zip")
    else:
        shutil.copytree(DRIVE_DATA, DATA_ROOT)
    # if the archive had a wrapper folder, point DATA_ROOT at the folder that holds train/ and val/
    hits = [p.parent for p in Path(DATA_ROOT).rglob("classmap.txt") if (p.parent / "train").is_dir()]
    assert len(hits) == 1, f"expected one dataset root, found {hits}"
    if hits[0] != Path(DATA_ROOT):
        tmp = "/content/_data_tmp"; shutil.move(str(hits[0]), tmp); shutil.rmtree(DATA_ROOT); shutil.move(tmp, DATA_ROOT)
    (Path(DATA_ROOT) / "_READY").touch()
sys.path.insert(0, REPO_DIR)
from detlib.data import ensure_data_yaml
print(open(ensure_data_yaml(DATA_ROOT)).read())
for s in ("train", "val"):
    print(s, len(list((Path(DATA_ROOT) / s / "images").iterdir())), "images")

In [ ]:
# Known-answer tests (scoring vs pycocotools, tiling, tile merge, OOM fallback). Must print 'tests passed'.
!DATA_ROOT={DATA_ROOT} python tests/test_pipeline.py

## 1-epoch timing probe (run before committing to full runs)
Trains B0 and B1 for 1 epoch each into `DRIVE_RUNS/_timing/` (separate from the real runs) and estimates 100-epoch wall time. Epoch 1 includes warm-up and first-time data loading, so the estimate is rough. **If B1 is estimated at more than ~8 h, STOP and decide the schedule before running the cells below.**

In [ ]:
!python train.py --config configs/b0.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS}/_timing --name b0_timing --epochs 1 --oom-fallback-batch 8
!python train.py --config configs/b1.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS}/_timing --work-dir {WORK_DIR} --name b1_timing --epochs 1 --oom-fallback-batch 8

In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for cfg, name in (("b0", "b0_timing"), ("b1", "b1_timing")):
    d = Path(DRIVE_RUNS) / "_timing" / name
    s = json.load(open(d / "training_summary.json"))
    r = pd.read_csv(d / "train" / "results.csv"); r.columns = [c.strip() for c in r.columns]
    sec = float(r["time"].iloc[-1]) if "time" in r else float("nan")  # seconds for epoch 1 (train + val)
    hw = json.load(open(d / "env" / "train_hardware.json"))
    rows.append(dict(run=cfg, gpu=", ".join(hw.get("gpus", [])), batch_used=s["batch_used"],
                     train_images_or_tiles=s["train_images_or_tiles"], iters_per_epoch=s["iterations_per_epoch"],
                     sec_epoch1=round(sec, 1), est_hours_100_epochs=round(sec * 100 / 3600, 2)))
    if cfg == "b1":
        print("B1 tiling:", json.load(open(d / "tiling_params.json"))["counts"])
t = pd.DataFrame(rows); print(t.to_string(index=False))
t.to_csv(Path(DRIVE_RUNS) / "_timing" / "timing_estimate.csv", index=False)
b1h = t.loc[t.run == "b1", "est_hours_100_epochs"].item()
print("\nSTOP: B1 estimate > 8 h, decide the schedule before full runs" if b1h > 8 else "\nB1 estimate within 8 h")

## B1 first: native-resolution 1024 tiles, 50 epochs. Re-run the train cell to resume after a disconnect.
Free-tier sessions cap at ~4h50m, so a run may need a second session: reconnect, re-run the setup cells, then the same train cell.
`last.pt` is saved every epoch and `epoch010.pt … epoch050.pt` are kept on Drive. If batch 16 runs out of GPU memory before the first epoch is saved, `train.py` retries at 8 and records it.

In [ ]:
!python train.py --config configs/b1.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS} --work-dir {WORK_DIR} --oom-fallback-batch 8

In [ ]:
!python eval.py --config configs/b1.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS}
!python analysis/merge_sensitivity.py --raw {DRIVE_RUNS}/b1_tile1024/eval/predictions_raw.csv --name b1_tile1024 --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures
!python analysis/pred_review.py --preds {DRIVE_RUNS}/b1_tile1024/eval/predictions.csv --name b1_tile1024 --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures
!python analysis/checkpoint_curve.py --config configs/b1.yaml --runs-root {DRIVE_RUNS} --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures

## B0 second: full image at 640, 50 epochs (reference). Re-run the train cell to resume after a disconnect.

In [ ]:
!python train.py --config configs/b0.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS} --oom-fallback-batch 8

In [ ]:
!python eval.py --config configs/b0.yaml --data-root {DATA_ROOT} --runs-root {DRIVE_RUNS} --ultra-crosscheck
!python analysis/pred_review.py --preds {DRIVE_RUNS}/b0_full640/eval/predictions.csv --name b0_full640 --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures
!python analysis/checkpoint_curve.py --config configs/b0.yaml --runs-root {DRIVE_RUNS} --data-root {DATA_ROOT} --out-root {DRIVE_RUNS}/figures

## Summary and export (everything except weights)

In [ ]:
import json, pandas as pd, shutil
from pathlib import Path
for run in ("b0_full640", "b1_tile1024"):
    d = Path(DRIVE_RUNS) / run
    if (d / "eval" / "per_class.csv").exists():
        print(f"== {run}")
        print(pd.read_csv(d / "eval" / "per_class.csv")[["class_name", "n_gt", "AP50_coco", "AP50_coco_ci95_lo", "AP50_coco_ci95_hi", "AP50_ultralytics"]].to_string(index=False))
        print("scorer comparison:", json.load(open(d / "eval" / "metrics.json")).get("scorer_comparison"))
    if (d / "training_summary.json").exists():
        print("training:", json.load(open(d / "training_summary.json")))
!cd {DRIVE_RUNS} && zip -qr /content/runs_export.zip . -x "*.pt"
shutil.copy("/content/runs_export.zip", str(Path(DRIVE_RUNS).parent / "runs_export.zip"))
print("export:", Path(DRIVE_RUNS).parent / "runs_export.zip", "(weights stay in", DRIVE_RUNS, ")")